In [ ]:
# !pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community langchain-chroma

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-proj-"

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# from langchain.vectorstores import Chroma. # This is the old import statement for Chroma, which is now replaced by the one below.
from langchain_chroma import Chroma

In [9]:
# from langchain.schema import Document # This is the old import statement for Document, which is now replaced by the one below.
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [10]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [11]:
doc1

Document(metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.')

In [ ]:
# creating vector store using chromadb, which will store embeddings of docs created above
vector_store = Chroma(  # instantiating a Chroma instance
    embedding_function=embeddings,
    persist_directory='my_chroma_db',  # creates a directory where files will be stored (has an sqlite3 file by default which it uses internally), can even not put this if just experimenting and don't want to persist data
    collection_name='sample' # chromadb internally organizes data in collections, so we need to give a name to the collection where all our docs "of this collection" will be stored. We can have multiple collections within the same database.
)

In [14]:
# add documents to the vector store, which will create embeddings for these docs and store them in the chromadb instance we created above
vector_store.add_documents(docs) # it also gives each doc a unique id

['7d98a6b9-74b6-44be-9666-9104b0a00105',
 '71a67cb0-d148-4f4a-90aa-f3830256cfb6',
 'f613678b-4760-49ce-891e-a133b647196c',
 'bd7a12dc-44ff-48b7-ad29-b5d0751d4987',
 '6fcb5f33-c89f-487f-9203-34a868492567']

In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7d98a6b9-74b6-44be-9666-9104b0a00105',
  '71a67cb0-d148-4f4a-90aa-f3830256cfb6',
  'f613678b-4760-49ce-891e-a133b647196c',
  'bd7a12dc-44ff-48b7-ad29-b5d0751d4987',
  '6fcb5f33-c89f-487f-9203-34a868492567'],
 'embeddings': array([[ 0.00560794, -0.02311153, -0.01429905, ...,  0.02093876,
         -0.01383779,  0.01245401],
        [-0.00122415, -0.03854982, -0.01535764, ...,  0.00106168,
         -0.00460609,  0.00201554],
        [ 0.06522571, -0.04097192, -0.01558281, ..., -0.00544899,
         -0.00448208, -0.01141575],
        [-0.00968686, -0.02968478, -0.01195252, ..., -0.0001826 ,
         -0.00672185,  0.01035153],
        [ 0.0091211 , -0.02845641, -0.01652192, ...,  0.0124004 ,
          0.00943171, -0.01972356]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [ ]:
# # to directly create a vector store and add given documents
# vector_store1 = Chroma.from_documents(  
#     documents=docs,
#     embedding_function=embeddings,
#     persist_directory='my_chroma_db',  
#     collection_name='sample' 
# )

In [16]:
# search document which is top k most similar to the query
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='bd7a12dc-44ff-48b7-ad29-b5d0751d4987', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='6fcb5f33-c89f-487f-9203-34a868492567', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [17]:
vector_store.similarity_search(
    query='Who among these is the best batsman?',
    k=1
)

[Document(id='7d98a6b9-74b6-44be-9666-9104b0a00105', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.')]

In [18]:
vector_store.similarity_search(
    query='Who among these is the best batsman?',
    k=2
)
# eh, Rohit Sharma content has nothing batsman related so fair that it doesnt return that, but instead of Bumrah should have given Jadeja as his content does have "all-rounder who contributes with both bat and ball"

[Document(id='7d98a6b9-74b6-44be-9666-9104b0a00105', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
 Document(id='bd7a12dc-44ff-48b7-ad29-b5d0751d4987', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [23]:
vector_store.similarity_search(
    query='Who among these is the best batsman?',
    k=2
)
# resampled 5 times and still gives same answer so idk how well this works, maybe because the content of docs is very small and not much info to differentiate them, so maybe with more content it will work better.

[Document(id='7d98a6b9-74b6-44be-9666-9104b0a00105', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
 Document(id='bd7a12dc-44ff-48b7-ad29-b5d0751d4987', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [24]:
# search with similarity (smaller is better as it is distance between query and doc)
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='bd7a12dc-44ff-48b7-ad29-b5d0751d4987', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.1294753551483154),
 (Document(id='6fcb5f33-c89f-487f-9203-34a868492567', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.3609352111816406)]

In [25]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='f613678b-4760-49ce-891e-a133b647196c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8512530326843262),
 (Document(id='6fcb5f33-c89f-487f-9203-34a868492567', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.910616397857666)]

In [26]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [27]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7d98a6b9-74b6-44be-9666-9104b0a00105',
  '71a67cb0-d148-4f4a-90aa-f3830256cfb6',
  'f613678b-4760-49ce-891e-a133b647196c',
  'bd7a12dc-44ff-48b7-ad29-b5d0751d4987',
  '6fcb5f33-c89f-487f-9203-34a868492567'],
 'embeddings': array([[ 0.00560794, -0.02311153, -0.01429905, ...,  0.02093876,
         -0.01383779,  0.01245401],
        [-0.00122415, -0.03854982, -0.01535764, ...,  0.00106168,
         -0.00460609,  0.00201554],
        [ 0.06522571, -0.04097192, -0.01558281, ..., -0.00544899,
         -0.00448208, -0.01141575],
        [-0.00968686, -0.02968478, -0.01195252, ..., -0.0001826 ,
         -0.00672185,  0.01035153],
        [ 0.0091211 , -0.02845641, -0.01652192, ...,  0.0124004 ,
          0.00943171, -0.01972356]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [28]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [29]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7d98a6b9-74b6-44be-9666-9104b0a00105',
  '71a67cb0-d148-4f4a-90aa-f3830256cfb6',
  'f613678b-4760-49ce-891e-a133b647196c',
  'bd7a12dc-44ff-48b7-ad29-b5d0751d4987',
  '6fcb5f33-c89f-487f-9203-34a868492567'],
 'embeddings': array([[ 0.00560794, -0.02311153, -0.01429905, ...,  0.02093876,
         -0.01383779,  0.01245401],
        [-0.00122415, -0.03854982, -0.01535764, ...,  0.00106168,
         -0.00460609,  0.00201554],
        [ 0.06522571, -0.04097192, -0.01558281, ..., -0.00544899,
         -0.00448208, -0.01141575],
        [-0.00968686, -0.02968478, -0.01195252, ..., -0.0001826 ,
         -0.00672185,  0.01035153],
        [ 0.0091211 , -0.02845641, -0.01652192, ...,  0.0124004 ,
          0.00943171, -0.01972356]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m